# Bulgaria – Data Driven Entrepreneurship | Individual Assignment Step 1
### Create **50 variables** (10 categories × 5 years: 2020–2024)

**The 10 variables per year (matching Belgium example format):**
1. `Growth YYYY` – % employee growth (year-over-year)
2. `aagr YYYY` – Annualised Average Growth Rate (3-year)
3. `Scaler YYYY`
4. `HighGrowthFirm YYYY`
5. `ConsistentHighGrowthFirm YYYY`
6. `VeryHighGrowthFirm YYYY`
7. `Gazelle YYYY`
8. `Mature YYYY`
9. `Scaleup YYYY`
10. `Superstar YYYY`

In [ ]:
import pandas as pd
import numpy as np
import os

print("Working directory:", os.getcwd())

# ⚠️ Make sure BG.xlsx is in the same folder as this notebook
# If not, replace with full path e.g. r"C:\Users\YourName\Downloads\BG.xlsx"
df = pd.read_excel("BG.xlsx")
fulldata = df[df["Country ISO code"] == "BG"].reset_index(drop=True)
print(f"Bulgaria companies loaded: {fulldata.shape[0]:,} rows")

## Variable 1/10 – Annual Employee Growth Rate

In [ ]:
def compute_growth(data, col_end, col_start):
    """(emp_end - emp_start) / emp_start. Returns 'n.a.' if missing or start==0."""
    result = []
    for i in range(len(data)):
        val_end   = data[col_end].iloc[i]
        val_start = data[col_start].iloc[i]
        if val_end == 'n.a.' or val_start == 'n.a.':
            result.append('n.a.')
        else:
            try:
                end = int(val_end); start = int(val_start)
                if start == 0:
                    result.append('n.a.')
                else:
                    result.append((end - start) / start)
            except:
                result.append('n.a.')
    return result

# 2018 & 2019 are intermediate — needed for the 3-year windows of 2020 & 2021
growth_pairs = [
    ('Number of employees\n2018', 'Number of employees\n2017', '_Growth 2018'),  # intermediate
    ('Number of employees\n2019', 'Number of employees\n2018', '_Growth 2019'),  # intermediate
    ('Number of employees\n2020', 'Number of employees\n2019', 'Growth 2020'),
    ('Number of employees\n2021', 'Number of employees\n2020', 'Growth 2021'),
    ('Number of employees\n2022', 'Number of employees\n2021', 'Growth 2022'),
    ('Number of employees\n2023', 'Number of employees\n2022', 'Growth 2023'),
    ('Number of employees\n2024', 'Number of employees\n2023', 'Growth 2024'),
]

for col_end, col_start, name in growth_pairs:
    fulldata[name] = compute_growth(fulldata, col_end, col_start)

print("Growth 2020–2024 created")

## Variable 2/10 – Annualised Average Growth Rate (aagr)

Formula: `aagr = ((emp_end / emp_start)^(1/3) - 1) × 100`  
⚠️ Requires **emp_start ≥ 10** (ESI Monitor criterion)

In [ ]:
def compute_aagr(data, col_end, col_start, years=3):
    """3-year annualised growth rate (%). Returns 'n.a.' if missing or start < 10."""
    result = []
    for i in range(len(data)):
        val_end   = data[col_end].iloc[i]
        val_start = data[col_start].iloc[i]
        if val_end == 'n.a.' or val_start == 'n.a.':
            result.append('n.a.')
        else:
            try:
                end = int(val_end); start = int(val_start)
                if start < 10:
                    result.append('n.a.')
                else:
                    result.append(((end / start) ** (1 / years) - 1) * 100)
            except:
                result.append('n.a.')
    return result

# Each focal year uses a 3-year lookback:
# aagr 2020 = 2017 → 2020
# aagr 2021 = 2018 → 2021
# aagr 2022 = 2019 → 2022
# aagr 2023 = 2020 → 2023
# aagr 2024 = 2021 → 2024
aagr_pairs = [
    ('Number of employees\n2020', 'Number of employees\n2017', 'aagr 2020'),
    ('Number of employees\n2021', 'Number of employees\n2018', 'aagr 2021'),
    ('Number of employees\n2022', 'Number of employees\n2019', 'aagr 2022'),
    ('Number of employees\n2023', 'Number of employees\n2020', 'aagr 2023'),
    ('Number of employees\n2024', 'Number of employees\n2021', 'aagr 2024'),
]

for col_end, col_start, name in aagr_pairs:
    fulldata[name] = compute_aagr(fulldata, col_end, col_start)

print("aagr 2020–2024 created")

## Variables 3–10/10 – Classification dummies

| Variable | Criteria |
|---|---|
| **Scaler** | aagr > 10%, emp_start ≥ 10 |
| **HighGrowthFirm** | aagr > 20%, emp_start ≥ 10 |
| **ConsistentHighGrowthFirm** | HGF + annual growth > 20% in ≥ 2/3 years |
| **VeryHighGrowthFirm** | HGF + annual growth > 40% in ≥ 2/3 years |
| **Gazelle** | ConsistentHGF + age ≤ 10 |
| **Mature** | ConsistentHGF + age > 10 |
| **Scaleup** | VeryHighGrowthFirm + age ≤ 10 |
| **Superstar** | VeryHighGrowthFirm + age > 10 |

In [ ]:
def make_classifications(data, focal_year):
    """
    For focal year FY:
      - 3-year annual growth window: Growth(FY-2), Growth(FY-1), Growth(FY)
      - emp_start = employees at FY-3
      - Age is computed internally (not saved as a column)
    """
    fy = focal_year
    aagr_col      = f'aagr {fy}'
    g_cols        = [f'_Growth {fy-2}' if fy-2 < 2020 else f'Growth {fy-2}',
                     f'_Growth {fy-1}' if fy-1 < 2020 else f'Growth {fy-1}',
                     f'Growth {fy}']
    emp_start_col = f'Number of employees\n{fy-3}'

    scalers, hgfs, cons_hgfs, very_hgfs = [], [], [], []
    gazelles, matures, scaleups, superstars = [], [], [], []

    for i in range(len(data)):
        aagr  = data[aagr_col].iloc[i]
        emp_s = data[emp_start_col].iloc[i] if emp_start_col in data.columns else 'n.a.'
        fy_val = data['Founded Year'].iloc[i]
        g_vals = [data[gc].iloc[i] if gc in data.columns else 'n.a.' for gc in g_cols]

        # Compute age internally
        age = (fy - int(fy_val)) if not pd.isna(fy_val) else 'n.a.'

        # Any missing → all n.a.
        if aagr == 'n.a.' or emp_s == 'n.a.' or age == 'n.a.' or any(v == 'n.a.' for v in g_vals):
            for lst in [scalers, hgfs, cons_hgfs, very_hgfs, gazelles, matures, scaleups, superstars]:
                lst.append('n.a.')
            continue

        try:
            emp_s_int = int(emp_s)
        except:
            for lst in [scalers, hgfs, cons_hgfs, very_hgfs, gazelles, matures, scaleups, superstars]:
                lst.append('n.a.')
            continue

        if emp_s_int < 10:
            for lst in [scalers, hgfs, cons_hgfs, very_hgfs, gazelles, matures, scaleups, superstars]:
                lst.append('n.a.')
            continue

        aagr_f   = float(aagr)
        g_floats = [float(v) for v in g_vals]
        age_i    = int(age)

        scaler   = 1 if aagr_f > 10 else 0
        hgf      = 1 if aagr_f > 20 else 0
        cons_hgf = 1 if (hgf == 1 and sum(g > 0.2 for g in g_floats) >= 2) else 0
        very_hgf = 1 if (hgf == 1 and sum(g > 0.4 for g in g_floats) >= 2) else 0
        gazelle  = 1 if (cons_hgf == 1 and age_i <= 10) else 0
        mature   = 1 if (cons_hgf == 1 and age_i >  10) else 0
        scaleup  = 1 if (very_hgf == 1 and age_i <= 10) else 0
        superstar= 1 if (very_hgf == 1 and age_i >  10) else 0

        scalers.append(scaler);    hgfs.append(hgf)
        cons_hgfs.append(cons_hgf); very_hgfs.append(very_hgf)
        gazelles.append(gazelle);  matures.append(mature)
        scaleups.append(scaleup);  superstars.append(superstar)

    data[f'Scaler {fy}']                     = scalers
    data[f'HighGrowthFirm {fy}']             = hgfs
    data[f'ConsistentHighGrowthFirm {fy}']   = cons_hgfs
    data[f'VeryHighGrowthFirm {fy}']         = very_hgfs
    data[f'Gazelle {fy}']                    = gazelles
    data[f'Mature {fy}']                     = matures
    data[f'Scaleup {fy}']                    = scaleups
    data[f'Superstar {fy}']                  = superstars
    return data


for yr in [2020, 2021, 2022, 2023, 2024]:
    fulldata = make_classifications(fulldata, yr)

print("All 8 classification dummies created for 2020–2024")

## ✅ Verification – All 50 variables present

In [ ]:
all_50_vars = []
for yr in [2020, 2021, 2022, 2023, 2024]:
    all_50_vars += [
        f'Growth {yr}',
        f'aagr {yr}',
        f'Scaler {yr}',
        f'HighGrowthFirm {yr}',
        f'ConsistentHighGrowthFirm {yr}',
        f'VeryHighGrowthFirm {yr}',
        f'Gazelle {yr}',
        f'Mature {yr}',
        f'Scaleup {yr}',
        f'Superstar {yr}',
    ]

print(f"Total variables: {len(all_50_vars)} (expected: 50)")
print()
for v in all_50_vars:
    status = '✅' if v in fulldata.columns else '❌ MISSING'
    print(f"  {status}  {v}")

## Summary table

In [ ]:
categories = ['Scaler', 'HighGrowthFirm', 'ConsistentHighGrowthFirm',
              'VeryHighGrowthFirm', 'Gazelle', 'Mature', 'Scaleup', 'Superstar']

summary = {}
for yr in [2020, 2021, 2022, 2023, 2024]:
    row = {}
    for cat in categories:
        col = f'{cat} {yr}'
        vals = fulldata[col]
        n_ones  = (vals == 1).sum()
        n_valid = ((vals == 1) | (vals == 0)).sum()
        pct = round(n_ones / n_valid * 100, 2) if n_valid > 0 else 0
        row[cat] = f"{n_ones} ({pct}%)"
    summary[yr] = row

summary_df = pd.DataFrame(summary).T
summary_df.index.name = 'Year'
summary_df

In [ ]:
# Drop intermediate columns (not part of the 50 variables)
fulldata.drop(columns=['_Growth 2018', '_Growth 2019'], inplace=True, errors='ignore')

# Save
fulldata.to_excel("BG_with_variables.xlsx", index=False)

print(f"✅ Saved: BG_with_variables.xlsx")
print(f"   Rows: {len(fulldata):,}  |  Total columns: {len(fulldata.columns)}")